# فحص GPU الفعلي لمدار 1.9.6
اختر **بيئة التشغيل ← تغيير نوع بيئة التشغيل ← T4 GPU** ثم **تشغيل الكل**.
ينفذ نفس عامل UVR المثبت في التطبيق على صوت صناعي بطول 17 ثانية ثم ثانية واحدة، ويفحص اكتمال المخرجات. لا يقرأ ملفاتك أو يطلب مفاتيح أو بيانات دخول. قد يستهلك وحدات حساب Colab.
أرسل التقرير الذي يظهر أسفل الخلية؛ يتضمن رمز خروج العملية وآخر سجل المحرك. هذا فحص تشخيصي؛ ليس وعدًا بإصلاح عطل غير مشخص.

In [ ]:
import os, sys, json, time, uuid, hashlib, zlib, signal, subprocess
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("اختر GPU من بيئة التشغيل ← تغيير نوع بيئة التشغيل، ثم أعد تشغيل الكل")
print("GPU:",torch.cuda.get_device_name(0),"Torch:",torch.__version__,"CUDA:",torch.version.cuda)
base=Path('/content') if Path('/content').exists() else Path.cwd()
service=base/('madar_'+uuid.uuid4().hex)
service.mkdir()
WORKER = '"""UVR Voc_FT with unchanged audio windows; one CUDA model per viewing session."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport os\nimport subprocess\nimport sys\nimport urllib.request\nimport time\nimport shutil\nimport re\nimport zlib\nimport threading\n\n_STATUS_LOCK = threading.RLock()\n_STATUS_PATH = None\n_STATUS = {}\n_HEARTBEAT = 0\n\nENGINE = None\nGPU_FREE_BEFORE = None\nGPU_VERIFIED = False\nCACHE = None\n\nRATE = 44100\nCORE = 4 * RATE\nMODEL_SHA = \'e411182ce2c53541fefdcc99a8f46f2fe03978eb22038b9497c1d9d95a00fad4\'\nMODEL_URL = \'https://github.com/k2-fsa/sherpa-onnx/releases/download/source-separation-models/UVR-MDX-NET-Voc_FT.onnx\'\n\n\ndef write_json(path, value):\n    p = Path(path)\n    tmp = p.with_suffix(\'.tmp\')\n    tmp.write_text(json.dumps(value), encoding=\'utf-8\')\n    tmp.replace(p)\n\n\ndef status(stage, progress, parts=None, frames=None, packs=None):\n    global _STATUS_PATH, _STATUS, _HEARTBEAT\n    value = dict(stage=stage, progress=progress)\n    if parts is not None:\n        value.update(chunks=parts, frames=frames, sample_rate=RATE, channels=2, model_sha256=MODEL_SHA)\n    if packs is not None:\n        value[\'packs\'] = packs\n    with _STATUS_LOCK:\n        _HEARTBEAT += 1\n        value[\'heartbeat\'] = _HEARTBEAT\n        _STATUS_PATH = Path(\'progress.json\').resolve()\n        _STATUS = value\n        write_json(_STATUS_PATH, value)\n\n\ndef heartbeat(stop, interval=2):\n    global _HEARTBEAT\n    while not stop.wait(interval):\n        with _STATUS_LOCK:\n            if _STATUS_PATH is not None and _STATUS.get(\'stage\') not in (\'failed\', \'done\'):\n                _HEARTBEAT += 1\n                _STATUS[\'heartbeat\'] = _HEARTBEAT\n                write_json(_STATUS_PATH, _STATUS)\n\n\ndef report_failure(error):\n    code = str(error) if isinstance(error, ValueError) and re.fullmatch(r\'[A-Z][A-Z0-9_]{0,79}\', str(error)) else type(error).__name__\n    with _STATUS_LOCK:\n        previous = _STATUS.get(\'stage\', \'bootstrap\')\n        path = _STATUS_PATH or Path(\'progress.json\').resolve()\n        write_json(path, dict(stage=\'failed\', last_stage=previous, progress=0, error=code))\n\n\ndef validate_request(req):\n    frames = req.get(\'frames\')\n    chunks = req.get(\'chunks\')\n    if type(frames) is not int or not 0 < frames <= 600 * RATE:\n        raise ValueError(\'MUSIC_DURATION\')\n    if not isinstance(chunks, list) or len(chunks) != (frames + CORE - 1) // CORE:\n        raise ValueError(\'MUSIC_CHUNKS\')\n    for i, part in enumerate(chunks):\n        if (not isinstance(part, dict) or part.get(\'bytes\') != min(CORE, frames - i * CORE) * 4\n                or not isinstance(part.get(\'sha256\'), str) or len(part[\'sha256\']) != 64):\n            raise ValueError(\'MUSIC_MANIFEST\')\n    return frames, chunks\n\n\ndef prepare():\n    global ENGINE, GPU_FREE_BEFORE, GPU_VERIFIED\n    if ENGINE is None:\n        status(\'gpu_check\', 27)\n        import torch\n        if not torch.cuda.is_available() or not torch.version.cuda:\n            raise ValueError(\'MUSIC_GPU_UNAVAILABLE\')\n        major = int(torch.version.cuda.split(\'.\')[0])\n        cuda = 13 if major >= 13 else 12\n        version = f\'1.13.8+cuda{cuda}.cudnn9.onnxruntime1.28.2\'\n        status(\'install\', 28)\n        from importlib.metadata import PackageNotFoundError, version as installed_version\n        try:\n            installed = installed_version(\'sherpa-onnx\') == version and installed_version(\'numpy\') == \'2.2.6\'\n        except PackageNotFoundError:\n            installed = False\n        if not installed:\n            with open(\'install.log\', \'wb\') as log:\n                subprocess.run([sys.executable, \'-m\', \'pip\', \'install\', \'--disable-pip-version-check\',\n                                \'--only-binary=:all:\', \'--timeout\', \'20\', \'--retries\', \'2\',\n                                f\'sherpa-onnx=={version}\', \'numpy==2.2.6\',\n                                \'-f\', \'https://k2-fsa.github.io/sherpa/onnx/cuda.html\'],\n                               stdout=log, stderr=log, check=True, timeout=600)\n        # Importing torch first loads the CUDA/cuDNN libraries used by this runtime.\n        import sherpa_onnx\n        from importlib.metadata import version as installed_version\n        if version != installed_version(\'sherpa-onnx\'):\n            raise ValueError(\'MUSIC_GPU_WHEEL\')\n        status(\'model_download\', 32)\n        model = (CACHE or Path.cwd()) / \'vocals.onnx\'\n        if not model.is_file() or hashlib.sha256(model.read_bytes()).hexdigest() != MODEL_SHA:\n            temporary = model.with_suffix(\'.download\')\n            digest = hashlib.sha256()\n            with urllib.request.urlopen(MODEL_URL, timeout=60) as response, temporary.open(\'wb\') as out:\n                size = 0\n                while True:\n                    data = response.read(1024 * 1024)\n                    if not data:\n                        break\n                    size += len(data)\n                    if size > 100 * 1024 * 1024:\n                        raise ValueError(\'MODEL_SIZE\')\n                    digest.update(data)\n                    out.write(data)\n            if digest.hexdigest() != MODEL_SHA:\n                temporary.unlink(missing_ok=True)\n                raise ValueError(\'MODEL_HASH\')\n            temporary.replace(model)\n        status(\'model_load\', 38)\n        GPU_FREE_BEFORE = torch.cuda.mem_get_info()[0]\n        config = sherpa_onnx.OfflineSourceSeparationConfig(\n            model=sherpa_onnx.OfflineSourceSeparationModelConfig(\n                uvr=sherpa_onnx.OfflineSourceSeparationUvrModelConfig(model=str(model)),\n                num_threads=2, provider=\'cuda\', debug=False))\n        if not config.validate():\n            raise ValueError(\'MODEL_CONFIG\')\n        ENGINE = sherpa_onnx.OfflineSourceSeparation(config)\n        status(\'ready\', 38)\n\n\ndef read_wire(part, name, expected):\n    encoding = part.get(\'wire_encoding\', \'pcm\')\n    if encoding not in (\'pcm\', \'zlib\'):\n        raise ValueError(\'MUSIC_WIRE_ENCODING\')\n    size = part.get(\'wire_bytes\') if encoding == \'zlib\' else expected\n    if type(size) is not int or not 0 < size <= expected or (encoding == \'zlib\' and size >= expected):\n        raise ValueError(\'MUSIC_WIRE_LENGTH\')\n    path = Path(name + (\'.zlib\' if encoding == \'zlib\' else \'\'))\n    if path.stat().st_size != size:\n        raise ValueError(\'MUSIC_WIRE_LENGTH\')\n    data = path.read_bytes()\n    if encoding == \'zlib\':\n        if hashlib.sha256(data).hexdigest() != part.get(\'wire_sha256\'):\n            raise ValueError(\'MUSIC_WIRE_HASH\')\n        try:\n            engine = zlib.decompressobj()\n            data = engine.decompress(data, expected + 1)\n            if len(data) != expected or not engine.eof or engine.unconsumed_tail or engine.unused_data:\n                raise ValueError(\'MUSIC_WIRE_LENGTH\')\n        except zlib.error as error:\n            raise ValueError(\'MUSIC_WIRE_HASH\') from error\n    if len(data) != expected or hashlib.sha256(data).hexdigest() != part.get(\'sha256\'):\n        raise ValueError(\'UPLOAD_HASH\')\n    path.unlink()\n    return data\n\n\ndef write_pack(name, data):\n    # Keep raw files for legacy clients; announce compressed bytes only when smaller.\n    Path(name).write_bytes(data)\n    part = dict(bytes=len(data), sha256=hashlib.sha256(data).hexdigest())\n    encoded = zlib.compress(data, level=1)\n    if len(encoded) + 512 < len(data):\n        Path(name + \'.zlib\').write_bytes(encoded)\n        part.update(wire_encoding=\'zlib\', wire_bytes=len(encoded), wire_sha256=hashlib.sha256(encoded).hexdigest())\n    return part\n\n\ndef run():\n    global GPU_VERIFIED\n    req = json.loads(Path(\'request.json\').read_text(\'utf-8\'))\n    frames, chunks = validate_request(req)\n    prepare()\n    import numpy as np\n    inputs = req.get(\'inputs\', chunks)\n    stride = 4 * CORE if \'inputs\' in req else CORE\n    if not isinstance(inputs, list) or len(inputs) != (frames + stride - 1) // stride:\n        raise ValueError(\'MUSIC_MANIFEST\')\n    with open(\'input.pcm\', \'wb\') as out:\n        for i, part in enumerate(inputs):\n            if not isinstance(part, dict) or part.get(\'bytes\') != min(stride, frames-i*stride)*4:\n                raise ValueError(\'MUSIC_MANIFEST\')\n            data = read_wire(part, f\'input_{i}.pcm\', min(stride, frames-i*stride)*4)\n            for at in range(0, len(data), CORE*4):\n                index = (i*stride + at//4) // CORE\n                block = data[at:at+CORE*4]\n                if hashlib.sha256(block).hexdigest() != chunks[index][\'sha256\']:\n                    raise ValueError(\'UPLOAD_HASH\')\n            out.write(data)\n    source = np.memmap(\'input.pcm\', dtype=\'<i2\', mode=\'r\', shape=(frames, 2))\n    parts, packs, packed = [], [], bytearray()\n    for i, start in enumerate(range(0, frames, CORE)):\n        left, right = max(0, start - RATE), min(frames, start + CORE + RATE)\n        samples = np.ascontiguousarray(source[left:right].T.astype(np.float32) / 32768)\n        output = ENGINE.process(sample_rate=RATE, samples=samples)\n        if not GPU_VERIFIED:\n            import torch\n            torch.cuda.synchronize()\n            # Detect a native provider silently falling back to CPU. This model\'s\n            # GPU weights alone exceed 8 MiB; no CPU result is labelled GPU.\n            if GPU_FREE_BEFORE - torch.cuda.mem_get_info()[0] < 8 * 1024 * 1024:\n                raise ValueError(\'MUSIC_GPU_PROVIDER\')\n            GPU_VERIFIED = True\n        if output.sample_rate != RATE or len(output.stems) != 2:\n            raise ValueError(\'MUSIC_OUTPUT_FORMAT\')\n        vocals = output.stems[0].data\n        keep, offset = min(CORE, frames - start), start - left\n        if vocals.ndim != 2 or vocals.shape[0] != 2 or vocals.shape[1] < offset + keep:\n            raise ValueError(\'MUSIC_OUTPUT_LENGTH\')\n        values = vocals[:, offset:offset + keep].T\n        if not np.isfinite(values).all():\n            raise ValueError(\'MUSIC_OUTPUT_FINITE\')\n        # Java Math.round(x) is floor(x + .5), including negative samples.\n        pcm = np.floor(np.clip(values, -1, 1) * 32767 + .5).astype(\'<i2\').tobytes()\n        Path(f\'output_{i}.pcm\').write_bytes(pcm)\n        parts.append(dict(bytes=len(pcm), sha256=hashlib.sha256(pcm).hexdigest()))\n        packed.extend(pcm)\n        if (i + 1) % 4 == 0 or start + CORE >= frames:\n            data = bytes(packed)\n            packs.append(write_pack(f\'pack_{len(packs)}.pcm\', data))\n            packed.clear()\n        status(\'separate\', 40 + int(54 * min(frames, start + CORE) / frames), parts, frames, packs)\n    write_json(\'result.json\', dict(frames=frames, sample_rate=RATE, channels=2,\n                                  model_sha256=MODEL_SHA, provider=\'cuda\', chunks=parts, packs=packs))\n    status(\'done\', 97, parts, frames, packs)\n\n\ndef run_job(folder):\n    os.chdir(folder)\n    try:\n        status(\'bootstrap\', 27)\n        run()\n    except Exception as error:\n        report_failure(error)\n        raise\n\n\ndef serve(root):\n    global CACHE\n    root = Path(root).resolve()\n    CACHE = root\n    os.chdir(root)\n    prepare()\n    previous = None\n    while True:\n        command = root / \'command.json\'\n        if not command.is_file():\n            time.sleep(.2)\n            continue\n        request = json.loads(command.read_text(\'utf-8\'))\n        folder = Path(request[\'folder\']).resolve()\n        if folder.parent != root.parent or not re.fullmatch(re.escape(root.name) + r\'_b[0-9]+\', folder.name):\n            raise ValueError(\'MUSIC_JOB_PATH\')\n        command.unlink()\n        # The caller sends the next command only after verifying the last result.\n        if previous is not None and previous != folder:\n            shutil.rmtree(previous)\n        run_job(folder)\n        previous = folder\n\n\nif __name__ == \'__main__\':\n    stop = threading.Event()\n    thread = threading.Thread(target=heartbeat, args=(stop,), daemon=True)\n    thread.start()\n    try:\n        if len(sys.argv) == 3 and sys.argv[2] == \'--serve\':\n            serve(sys.argv[1])\n        else:\n            run_job(Path(sys.argv[1]).resolve())\n    except Exception as error:\n        report_failure(error)\n        raise\n    finally:\n        stop.set()\n        thread.join(timeout=3)\n'
(service/'music_worker.py').write_text(WORKER)
report={'app_source':'MadarFiles 1.9.6 / 77c0171','torch':str(torch.__version__),'cuda':str(torch.version.cuda),'gpu':torch.cuda.get_device_name(0),'jobs':[]}
started=time.monotonic()
log=(service/'worker.log').open('wb')
process=subprocess.Popen([sys.executable,str(service/'music_worker.py'),str(service),'--serve'],stdout=log,stderr=log,start_new_session=True)
def state(path):
    try:return json.loads(path.read_text())
    except (OSError,ValueError):return {}
def wait_state(path, terminal, seconds):
    deadline=time.monotonic()+seconds
    previous=None
    while True:
        value=state(path)
        stage=value.get('stage')
        if stage!=previous and stage:
            print(round(time.monotonic()-started,1),'s',stage,flush=True);previous=stage
        if stage in terminal:return value
        exit_code=process.poll()
        if exit_code is not None:
            value=state(path) # Preserve a failure published just before exit.
            if value.get('stage') in terminal:return value
            raise RuntimeError('Worker exited: '+str(exit_code))
        if time.monotonic()>deadline:raise TimeoutError('No completion at stage '+str(stage))
        time.sleep(.2)
try:
    warm=wait_state(service/'progress.json',{'ready','failed'},900)
    report['warm']=warm;report['warm_seconds']=round(time.monotonic()-started,2)
    if warm.get('stage')!='ready':raise RuntimeError('Warmup failed')
    import numpy as np
    for number, seconds in enumerate([17,1]):
        job=base/(service.name+'_b'+str(number));job.mkdir()
        rate=44100;core=4*rate;frames=seconds*rate
        t=np.arange(frames,dtype=np.float64)/rate
        wave=(.12*np.sin(2*np.pi*220*t)+.07*np.sin(2*np.pi*997*t))
        mono=np.round(wave*32767).astype('<i2')
        raw=np.column_stack((mono,mono)).astype('<i2').tobytes()
        digest=lambda data:hashlib.sha256(data).hexdigest()
        chunks=[{'bytes':len(raw[i:i+core*4]),'sha256':digest(raw[i:i+core*4])} for i in range(0,len(raw),core*4)]
        inputs=[]
        for i,at in enumerate(range(0,len(raw),4*core*4)):
            block=raw[at:at+4*core*4];wire=zlib.compress(block,level=1)
            item={'bytes':len(block),'sha256':digest(block)};suffix=''
            if len(wire)+512<len(block):
                item.update(wire_encoding='zlib',wire_bytes=len(wire),wire_sha256=digest(wire));suffix='.zlib'
            else:wire=block
            (job/f'input_{i}.pcm{suffix}').write_bytes(wire);inputs.append(item)
        (job/'request.json').write_text(json.dumps({'frames':frames,'chunks':chunks,'inputs':inputs}))
        began=time.monotonic();command=service/'command.tmp'
        command.write_text(json.dumps({'folder':str(job)}));command.replace(service/'command.json')
        result=wait_state(job/'progress.json',{'done','failed'},180)
        entry={'seconds':seconds,'elapsed':round(time.monotonic()-began,2),'state':result};report['jobs'].append(entry)
        if result.get('stage')!='done':raise RuntimeError('GPU inference failed')
        manifest=state(job/'result.json');assert manifest.get('frames')==frames
        for i,part in enumerate(manifest['chunks']):
            output=(job/f'output_{i}.pcm').read_bytes();assert len(output)==part['bytes'] and digest(output)==part['sha256']
        assert sum(x['bytes'] for x in manifest['chunks'])==frames*4
        entry['verified']=True;print('Verified complete GPU output:',seconds,'seconds')
    report['result']='PASS_ACTUAL_GPU_INFERENCE'
except Exception as error:
    report['result']='FAILED';report['exception']=type(error).__name__;report['detail']=str(error)[:300]
finally:
    report['exit_code_before_cleanup']=process.poll()
    report['last_root_state']=state(service/'progress.json')
    log.flush()
    with (service/'worker.log').open('rb') as trace:
        trace.seek(max(0,(service/'worker.log').stat().st_size-6000));tail=trace.read().decode('utf-8','replace')
    report['worker_log_tail']=tail
    if process.poll() is None:
        os.killpg(process.pid,signal.SIGTERM)
        try:process.wait(timeout=5)
        except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait(timeout=5)
    log.close()
    text=json.dumps(report,ensure_ascii=False,indent=2)
    (service/'gpu-diagnostic-report.json').write_text(text)
    print('\n===== انسخ التقرير التالي كاملًا =====\n'+text)
